In [1]:
!pip install gensim nltk scikit-learn

In [2]:
import numpy as np
import nltk
import re
from nltk.corpus import stopwords
from gensim.downloader import load
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Soham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
word_vectors = load("glove-wiki-gigaword-100")

print("Embedding loaded. Vector size:", word_vectors.vector_size)

[==================================================] 100.0% 128.1/128.1MB downloaded
Embedding loaded. Vector size: 100


In [5]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    return [w for w in words if w not in stop_words]

In [6]:
def sentence_embedding(sentence):
    words = preprocess(sentence)
    vectors = []
    
    for word in words:
        if word in word_vectors:
            vectors.append(word_vectors[word])
    
    if len(vectors) == 0:
        return np.zeros(word_vectors.vector_size)
    
    return np.mean(vectors, axis=0)

In [7]:
intents = {
    "Account Issue": [
        "forgot password",
        "cannot login",
        "reset my password",
        "login problem"
    ],
    "Billing Issue": [
        "payment failed",
        "money deducted",
        "transaction issue",
        "refund not received"
    ],
    "Technical Issue": [
        "app crashes",
        "bug in application",
        "system not working",
        "error while opening app"
    ]
}

In [8]:
intent_vectors = {}

for intent, phrases in intents.items():
    vectors = [sentence_embedding(p) for p in phrases]
    intent_vectors[intent] = np.mean(vectors, axis=0)

In [9]:
def classify_query(query):
    query_vec = sentence_embedding(query)
    
    best_intent = None
    max_score = -1
    
    for intent, vec in intent_vectors.items():
        score = cosine_similarity([query_vec], [vec])[0][0]
        
        if score > max_score:
            max_score = score
            best_intent = intent
    
    return best_intent, max_score

In [10]:
queries = [
    "I can't access my account",
    "Payment got deducted but failed",
    "App keeps crashing on startup"
]

for q in queries:
    intent, score = classify_query(q)
    print(f"Query: {q}")
    print(f"Predicted Intent: {intent} (Similarity: {score:.2f})\n")

Query: I can't access my account
Predicted Intent: Billing Issue (Similarity: 0.63)

Query: Payment got deducted but failed
Predicted Intent: Billing Issue (Similarity: 0.92)

Query: App keeps crashing on startup
Predicted Intent: Technical Issue (Similarity: 0.75)



In [11]:
documents = [
    "How to reset password",
    "Steps for refund process",
    "Fix app crashing issue",
    "Login troubleshooting guide"
]

doc_vectors = [sentence_embedding(doc) for doc in documents]

def semantic_search(query):
    query_vec = sentence_embedding(query)
    
    scores = cosine_similarity([query_vec], doc_vectors)[0]
    best_index = np.argmax(scores)
    
    return documents[best_index], scores[best_index]

# Test
result, score = semantic_search("I forgot my password")
print("Best Match:", result)
print("Similarity Score:", score)

Best Match: How to reset password
Similarity Score: 0.78626096
